In [26]:

import pandas as pd

def read_and_clean_data(file_path, sheet_name = 1):
    df = pd.read_excel(file_path, sheet_name = sheet_name)

    #df = df.iloc[1: , 1:]

    df = df.fillna(0)

    #print(df)
    return df


def create_dictionaries(df):
    dictionary = {}

    dictionary1 = {}

    for i in range(1, 99):
        row_key = int(df.iloc[i, 0])
        
        dictionary[row_key] = [] 
        dictionary1[row_key] = set()
        
        for j in range(1, 99):
            dictionary[row_key].append((df.iloc[0, j], df.iloc[i, j]))

            if df.iloc[i, j] <= 900:
                dictionary1[row_key].add(df.iloc[0, j])

    return dictionary, dictionary1


df = read_and_clean_data("Data assignment ambulance 2 Small(1).xlsx", sheet_name = 1)
dictionary, dictionary1 = create_dictionaries(df)

#print(dictionary)
print(dictionary1)

#for key, value in dictionary1.items():
 #   print((key,len(value)))


#for values in dictionary[1117.0]:
 #   print (values[1])









{1117: {1161.0, 1165.0, 2064.0, 2065.0, 1171.0, 1175.0, 1435.0, 1436.0, 1437.0, 1438.0, 2131.0, 2132.0, 2133.0, 2141.0, 1117.0, 1118.0, 1119.0, 2143.0, 2155.0, 2033.0, 2035.0}, 1118: {1161.0, 2064.0, 2065.0, 1171.0, 1175.0, 1435.0, 1436.0, 1437.0, 1438.0, 2103.0, 2131.0, 2132.0, 2133.0, 2134.0, 2135.0, 2136.0, 1117.0, 1118.0, 1119.0, 2141.0, 2142.0, 2143.0, 2151.0, 2152.0, 2153.0, 2154.0, 2155.0, 2156.0, 2157.0, 2033.0, 2035.0, 2165.0}, 1119: {1161.0, 1165.0, 2064.0, 2065.0, 1171.0, 1175.0, 1435.0, 1436.0, 1437.0, 1438.0, 2101.0, 2102.0, 2103.0, 2104.0, 2131.0, 2132.0, 2133.0, 2134.0, 2135.0, 2136.0, 1117.0, 1118.0, 1119.0, 2141.0, 2142.0, 2143.0, 2151.0, 2152.0, 2153.0, 2154.0, 2155.0, 2156.0, 2157.0, 2158.0, 2032.0, 2033.0, 2035.0, 2037.0, 2165.0}, 1161: {1161.0, 1165.0, 2061.0, 2063.0, 2064.0, 2065.0, 1171.0, 1941.0, 1942.0, 1943.0, 1175.0, 2142.0, 1946.0, 1947.0, 1948.0, 1437.0, 1436.0, 1951.0, 2143.0, 2082.0, 1971.0, 2101.0, 2102.0, 2103.0, 2106.0, 1981.0, 2111.0, 1985.0, 1991.0, 

In [27]:
#the algorithm to find the minimum number of ambulances needed to cover all the areas within 15 minutes

def greedy_algorithm(dictionary1, dictionary, num_ambulances = 2):
    seen = set()

    PD = -1

    max = 0

    PD_list = []

    for i in range(num_ambulances):
        for key, value in dictionary1.items():

            new = len(value - seen)
       

            if new > max:
                # implement something if they can reach the same amount of areas.
                max = new
                PD = key
            elif new == max:
                if PD == -1:
                    PD = key
                else:
                    total1 = 0
                    total2 = 0

                    for pc, time in dictionary[key]:
                        
                        total1 = total1 + time

                    for pc, time in dictionary[PD]:
                    
                        total2 = total2 + time

                    if total1 < total2:
                        PD = key    
        seen = seen.union(dictionary1[PD])
        PD_list.append(PD)
        max = 0
    return PD_list, seen


PD_list, seen = greedy_algorithm(dictionary1, dictionary, num_ambulances = 2)
print(PD_list)

        
        



[2024, 2131]


In [29]:
#second algorithm: take demand per postal code into account

def create_demand_dictionary(file_path, sheet_name = 2):
    demand_df = read_and_clean_data(file_path, sheet_name = sheet_name)
    
    demand_dictionary = {}
    
    for i in range(0, len(demand_df)):
        postal_code = int(demand_df.iloc[i, 0])
        demand = float(demand_df.iloc[i, 1])
        demand_dictionary[postal_code] = demand
    
    return demand_dictionary


def greedy_algorithm_with_demand(dictionary1, dictionary, demand_dictionary, num_ambulances = 2):
    seen = set()

    PD = -1

    max = 0

    PD_list = []

    for i in range(num_ambulances):
        for key, value in dictionary1.items():

            uncovered = value - seen
            weighted_new = sum(demand_dictionary.get(int(pc), 0) for pc in uncovered)
       

            if weighted_new > max:
                max = weighted_new
                PD = key
            elif weighted_new == max:
                if PD == -1:
                    PD = key
                else:
                    total1 = 0
                    total2 = 0

                    for pc, time in dictionary[key]:
                        
                        total1 = total1 + time

                    for pc, time in dictionary[PD]:
                        
                        total2 = total2 + time

                    if total1 < total2:
                        PD = key    
        seen = seen.union(dictionary1[PD])
        PD_list.append(PD)
        max = 0
    return PD_list, seen


demand_dictionary = create_demand_dictionary("Data assignment ambulance 2 Small(1).xlsx", sheet_name = 2)
PD_list_demand, seen_demand = greedy_algorithm_with_demand(dictionary1, dictionary, demand_dictionary, num_ambulances = 2)
print(PD_list_demand)

total_covered_demand = 0
for pc in seen_demand:
    postal_code = int(pc)
    demand_value = demand_dictionary.get(postal_code, 0)
    total_covered_demand = total_covered_demand + demand_value

print(total_covered_demand)


[2024, 2131]
519457.0


In [35]:
import random
import itertools

# build a random test instance with the same structure as your algorithms expect
def generate_random_instance(num_postal_codes = 20, edge_probability = 0.35):
    postal_codes = [1000 + i for i in range(num_postal_codes)]

    dictionary = {}
    dictionary1 = {}
    demand_dictionary = {}

    for pc in postal_codes:
        demand_dictionary[pc] = random.randint(1, 500)

    for station in postal_codes:
        dictionary[station] = []
        dictionary1[station] = set()

        for target in postal_codes:
            if station == target:
                travel_time = random.randint(60, 300)
            else:
                if random.random() < edge_probability:
                    travel_time = random.randint(300, 900)
                else:
                    travel_time = random.randint(901, 1800)

            dictionary[station].append((target, float(travel_time)))

            if travel_time <= 900:
                dictionary1[station].add(target)

    return dictionary, dictionary1, demand_dictionary


def brute_force_optimal(dictionary1, num_ambulances = 2, demand_dictionary = None):
    candidates = list(dictionary1.keys())
    k = min(num_ambulances, len(candidates))

    best_choice = None
    best_score = -1

    for choice in itertools.combinations(candidates, k):
        covered = set()
        for pd in choice:
            covered = covered.union(dictionary1[pd])

        if demand_dictionary is None:
            score = len(covered)
        else:
            score = 0
            for pc in covered:
                score = score + demand_dictionary.get(int(pc), 0)

        if score > best_score:
            best_score = score
            best_choice = list(choice)

    return best_choice, best_score


def run_sanity_tests():
    dictionary1_small = {
        1: set([1, 2]),
        2: set([2, 3]),
        3: set([3])
    }
    dictionary_small = {
        1: [(1, 100.0), (2, 100.0), (3, 1200.0)],
        2: [(1, 1200.0), (2, 100.0), (3, 100.0)],
        3: [(1, 1200.0), (2, 1200.0), (3, 100.0)]
    }
    demand_small = {1: 10, 2: 5, 3: 1}

    PD_list_test, seen_test = greedy_algorithm(dictionary1_small, dictionary_small, num_ambulances = 1)
    assert len(seen_test) == 2

    PD_list_demand_test, seen_demand_test = greedy_algorithm_with_demand(
        dictionary1_small, dictionary_small, demand_small, num_ambulances = 1
    )
    demand_score = 0
    for pc in seen_demand_test:
        demand_score = demand_score + demand_small.get(int(pc), 0)
    assert demand_score == 15

    print("Sanity tests passed")


def run_accuracy_experiment(
    num_cases = 100,
    num_postal_codes = 20,
    num_ambulances = 2,
    edge_probability = 0.35,
    random_seed = 42
    ):
    random.seed(random_seed)

    greedy1_exact_matches = 0
    greedy2_exact_matches = 0
    greedy1_ratios = []
    greedy2_ratios = []

    for case_id in range(num_cases):
        dictionary_test, dictionary1_test, demand_test = generate_random_instance(
            num_postal_codes = num_postal_codes,
            edge_probability = edge_probability
        )

        PD_list_1, seen_1 = greedy_algorithm(
            dictionary1_test, dictionary_test, num_ambulances = num_ambulances
        )
        greedy1_score = len(seen_1)

        best_choice_1, optimal1_score = brute_force_optimal(
            dictionary1_test, num_ambulances = num_ambulances, demand_dictionary = None
        )

        ratio1 = greedy1_score / optimal1_score
        greedy1_ratios.append(ratio1)

        if greedy1_score == optimal1_score:
            greedy1_exact_matches = greedy1_exact_matches + 1

        PD_list_2, seen_2 = greedy_algorithm_with_demand(
            dictionary1_test, dictionary_test, demand_test, num_ambulances = num_ambulances
        )
        greedy2_score = 0
        for pc in seen_2:
            greedy2_score = greedy2_score + demand_test.get(int(pc), 0)

        best_choice_2, optimal2_score = brute_force_optimal(
            dictionary1_test, num_ambulances = num_ambulances, demand_dictionary = demand_test
        )

        ratio2 = greedy2_score / optimal2_score
        greedy2_ratios.append(ratio2)

        if greedy2_score == optimal2_score:
            greedy2_exact_matches = greedy2_exact_matches + 1

    results = {
        "cases": num_cases,
        "nodes": num_postal_codes,
        "ambulances": num_ambulances,
        "greedy1_exact_match_rate": greedy1_exact_matches / num_cases,
        "greedy1_average_ratio": sum(greedy1_ratios) / len(greedy1_ratios),
        "greedy1_worst_ratio": min(greedy1_ratios),
        "greedy2_exact_match_rate": greedy2_exact_matches / num_cases,
        "greedy2_average_ratio": sum(greedy2_ratios) / len(greedy2_ratios),
        "greedy2_worst_ratio": min(greedy2_ratios)
    }
    return results


run_sanity_tests()

print("100-case experiment")
results_100 = run_accuracy_experiment(num_cases = 100, num_postal_codes = 20, num_ambulances = 2)
print(results_100)

print("1000-case experiment")
results_1000 = run_accuracy_experiment(num_cases = 1000, num_postal_codes = 20, num_ambulances = 2)
print(results_1000)

Sanity tests passed
100-case experiment
{'cases': 100, 'nodes': 20, 'ambulances': 2, 'greedy1_exact_match_rate': 0.74, 'greedy1_average_ratio': 0.9830682834537323, 'greedy1_worst_ratio': 0.8823529411764706, 'greedy2_exact_match_rate': 0.56, 'greedy2_average_ratio': 0.9852274411513264, 'greedy2_worst_ratio': 0.9087442806304016}
1000-case experiment
{'cases': 1000, 'nodes': 20, 'ambulances': 2, 'greedy1_exact_match_rate': 0.797, 'greedy1_average_ratio': 0.9866909786721707, 'greedy1_worst_ratio': 0.8421052631578947, 'greedy2_exact_match_rate': 0.616, 'greedy2_average_ratio': 0.9851284080284999, 'greedy2_worst_ratio': 0.8486356615754247}


In [ ]:
import time

start = time.time()
quick_results = run_accuracy_experiment(num_cases = 10, num_postal_codes = 100, num_ambulances = 2)
end = time.time()

print(quick_results)
print("elapsed_seconds", end - start)